In [1]:
# imports
import tensorflow as tf
from tensorflow import keras as keras
import matplotlib.pyplot as plt
import keras_tuner as kt
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import numpy as np

2024-05-22 18:02:16.305627: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-22 18:02:16.305688: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-05-22 18:02:16.306661: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-05-22 18:02:16.311854: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-22 18:02:16.969482: W tensorflow/compiler/tf2

In [2]:
#import  data

IMG_SIZE = 224
BATCH_SIZE = 32

train_data = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    shuffle = True,
    image_size = (IMG_SIZE,IMG_SIZE),
    batch_size = BATCH_SIZE
)

test_data = tf.keras.utils.image_dataset_from_directory(
    directory = "data/test",
    shuffle = True,
    image_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE
)

val_data = tf.keras.utils.image_dataset_from_directory(
    directory = "data/valid",
    shuffle = True,
    image_size = (IMG_SIZE,IMG_SIZE),
    batch_size = BATCH_SIZE
)

data_augmentation_layers = [
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    #tf.keras.layers.RandomZoom(0.1),
    #tf.keras.layers.RandomContrast(0.1),
]

def  data_augmentation(images):
    for layer in  data_augmentation_layers:
        images = layer(images)
    return images

def preprocess_image(image, label):
    image = tf.cast(image, tf.float32) / 255.0  # Normalize pixel values to [0, 1]
    image = data_augmentation(image)
    return image, label

train_data = train_data.map(
    preprocess_image,
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

val_data = val_data.map(
    lambda img, label: (tf.cast(img, tf.float32) / 255.0, label),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

test_data = test_data.map(
    lambda img, label: (tf.cast(img, tf.float32) / 255.0, label),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)


Found 2339 files belonging to 10 classes.


2024-05-22 18:02:18.540166: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-22 18:02:18.564247: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-22 18:02:18.564302: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-22 18:02:18.566857: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-22 18:02:18.566918: I external/local_xla/xla/stream_executor

Found 50 files belonging to 10 classes.
Found 50 files belonging to 10 classes.


In [3]:
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

IMG_SIZE = 224
BATCH_SIZE = 32
CHANNELS = 3
input_shape = (IMG_SIZE, IMG_SIZE, CHANNELS)

def HModel():
    model = keras.Sequential([
        
        keras.layers.Conv2D(32,kernel_size=3,input_shape=input_shape),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Flatten(),
        #keras.layers.Dropout(0.2),
        keras.layers.Dense(units = 256, activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(10, activation='softmax', activity_regularizer = tf.keras.regularizers.L2()),
    ])
    
    #, activity_regularizer = tf.keras.regularizers.L1(0.007)
    
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate = 0.00135),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits =False),
        metrics=["accuracy"],
    )
    return model


model = HModel()

lr_schedule_Plateau = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.15,
                              patience=5, min_lr=0.000001)


earlystopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=30, restore_best_weights=True)
#
#
history = model.fit(
train_data,
epochs = 75,
batch_size = 32,
validation_data = val_data,callbacks=[earlystopping,lr_schedule_Plateau])

#best learning rate:0.002113314496627678
#best regularization:0.007189638437333526

Epoch 1/75


2024-05-22 18:02:23.680492: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2024-05-22 18:02:23.840396: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-05-22 18:02:25.607134: I external/local_xla/xla/service/service.cc:168] XLA service 0x7ffa28006260 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-05-22 18:02:25.607178: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2024-05-22 18:02:25.611335: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1716393745.666413  361429 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


74/74 [==============================] - 13s 86ms/step - loss: 2.0667 - accuracy: 0.3142 - val_loss: 4.4624 - val_accuracy: 0.1000 - lr: 0.0014
Epoch 2/75
74/74 [==============================] - 3s 44ms/step - loss: 1.5903 - accuracy: 0.4459 - val_loss: 6.0312 - val_accuracy: 0.1000 - lr: 0.0014
Epoch 3/75
74/74 [==============================] - 3s 43ms/step - loss: 1.4523 - accuracy: 0.4681 - val_loss: 4.9910 - val_accuracy: 0.1400 - lr: 0.0014
Epoch 4/75
74/74 [==============================] - 3s 43ms/step - loss: 1.2615 - accuracy: 0.5468 - val_loss: 3.9124 - val_accuracy: 0.1800 - lr: 0.0014
Epoch 5/75
74/74 [==============================] - 3s 44ms/step - loss: 1.1686 - accuracy: 0.5836 - val_loss: 4.3246 - val_accuracy: 0.2000 - lr: 0.0014
Epoch 6/75
74/74 [==============================] - 3s 43ms/step - loss: 1.0551 - accuracy: 0.6165 - val_loss: 1.3283 - val_accuracy: 0.5600 - lr: 0.0014
Epoch 7/75
74/74 [==============================] - 3s 43ms/step - loss: 1.0162 - accu

KeyboardInterrupt: 

In [ ]:
#Plotting Training and Validation Metrics
# Plotting the accuracy
plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='val accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.title('Model Accuracy')
plt.show()

# Plotting the loss
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.title('Model Loss')
plt.show()

In [ ]:
#Evaluating the Model on the Test Set
model.evaluate(test_data, return_dict = True)
model.save(filepath = "model_88.keras")

Y_pred = model.predict(test_data)
y_pred = np.argmax(Y_pred, axis=1)

# Getting the true labels
y_true = test_data.classes

# Classification report
print('Classification Report')
print(classification_report(y_true, y_pred, target_names=test_data.class_indices.keys()))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print('Confusion Matrix')
print(cm)

# F1 score
f1 = f1_score(y_true, y_pred, average='weighted')
print(f'F1 Score: {f1:.2f}')

# Plotting the confusion matrix
import seaborn as sns

plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=test_data.class_indices.keys(), yticklabels=test_data.class_indices.keys())
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


In [3]:
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

IMG_SIZE = 224
BATCH_SIZE = 32
CHANNELS = 3
input_shape = (IMG_SIZE, IMG_SIZE, CHANNELS)

def HModel():
    model = keras.Sequential([
        
        keras.layers.Conv2D(32,kernel_size=3,input_shape=input_shape),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        #keras.layers.Dropout(0.2),
        
        keras.layers.Flatten(),
        #keras.layers.Dropout(0.2),
        keras.layers.Dense(units = 256, activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(10, activation='softmax', activity_regularizer = tf.keras.regularizers.L2()),
    ])
    
    #, activity_regularizer = tf.keras.regularizers.L1(0.007)
    
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate = 0.0015),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits =False),
        metrics=["accuracy"],
    )
    return model


model = HModel()

lr_schedule_Plateau = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.02,
                              patience=5, min_lr=0.0001)


earlystopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=20, restore_best_weights=True)
#
#
history = model.fit(
train_data,
epochs = 50,
batch_size = 32,
validation_data = val_data,callbacks=[earlystopping,lr_schedule_Plateau])

Epoch 1/50


2024-05-22 16:48:05.433116: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2024-05-22 16:48:05.597110: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-05-22 16:48:07.488493: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f115cb0de00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-05-22 16:48:07.488533: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2024-05-22 16:48:07.492509: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1716389287.548490  263176 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


74/74 [==============================] - 13s 87ms/step - loss: 1.9700 - accuracy: 0.3283 - val_loss: 4.9297 - val_accuracy: 0.0800 - lr: 0.0015
Epoch 2/50
74/74 [==============================] - 4s 47ms/step - loss: 1.5176 - accuracy: 0.4519 - val_loss: 4.9005 - val_accuracy: 0.1000 - lr: 0.0015
Epoch 3/50
74/74 [==============================] - 4s 44ms/step - loss: 1.3867 - accuracy: 0.5066 - val_loss: 3.4581 - val_accuracy: 0.0600 - lr: 0.0015
Epoch 4/50
74/74 [==============================] - 4s 45ms/step - loss: 1.3155 - accuracy: 0.5109 - val_loss: 2.0456 - val_accuracy: 0.3400 - lr: 0.0015
Epoch 5/50
74/74 [==============================] - 4s 47ms/step - loss: 1.1872 - accuracy: 0.5652 - val_loss: 1.9405 - val_accuracy: 0.2800 - lr: 0.0015
Epoch 6/50
74/74 [==============================] - 4s 45ms/step - loss: 1.0885 - accuracy: 0.6003 - val_loss: 2.0624 - val_accuracy: 0.3000 - lr: 0.0015
Epoch 7/50
74/74 [==============================] - 4s 44ms/step - loss: 1.1751 - accu

KeyboardInterrupt: 

In [ ]:
model.evaluate(val_data, return_dict = True)

2/2 [==============================] - 0s 12ms/step - loss: 0.6329 - accuracy: 0.7600


{'loss': 0.6329250931739807, 'accuracy': 0.7599999904632568}

In [ ]:
model.evaluate(test_data, return_dict = True)

2/2 [==============================] - 0s 13ms/step - loss: 0.3281 - accuracy: 0.8800


{'loss': 0.32810062170028687, 'accuracy': 0.8799999952316284}

In [ ]:
#model.save(filepath = "model_1.keras")